# ASR Lattice-Based Evaluation

Traditional Word Error Rate (WER) evaluation compares model output against a single ground truth string, which unfairly penalizes valid transcription variations. This notebook implements a **Lattice-based evaluation** approach that captures lexical, phonetical, and spelling variations in sequential "bins" to ensure fairer model assessment.

## 1. Setup and Data Loading

We load the dataset containing transcriptions from one human reference and six ASR models.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter

csv_path = '/content/Question 4 - Task.csv'
df = pd.read_csv(csv_path)
model_columns = ['Model H', 'Model i', 'Model k', 'Model l', 'Model m', 'Model n']
print(f"Loaded {len(df)} segments.")

Loaded 46 segments.


## 2. Text Normalization

Consistent naming and formatting are critical for word-level alignment.

In [2]:
def normalize_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[\.\,|।"\!\?\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for col in ['Human'] + model_columns:
    df[col] = df[col].apply(normalize_text)

## 3. Sequence Alignment Logic

We use the Edit Distance algorithm to align strings, enabling us to identify which words correspond to each other across different transcription outputs.

In [3]:
def get_edit_matrix(s1, s2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n + 1, m + 1))
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp

def align_pair(s1, s2):
    dp = get_edit_matrix(s1, s2)
    res1, res2 = [], []
    i, j = len(s1), len(s2)

    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + (0 if s1[i-1] == s2[j-1] else 1):
            res1.append(s1[i-1])
            res2.append(s2[j-1])
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            res1.append(s1[i-1])
            res2.append("<eps>")
            i -= 1
        else:
            res1.append("<eps>")
            res2.append(s2[j-1])
            j -= 1
    return res1[::-1], res2[::-1]

## 4. Lattice Construction and Trust Mechanism

We build a list of sequential "bins" based on the human reference. If multiple models (at least 3) agree on a word that disagrees with the reference, we assume the reference might be wrong and add the model consensus to the lattice.

In [4]:
def build_bins(row, threshold=3):
    human_words = row['Human'].split()
    bins = [[word] for word in human_words]

    for model in model_columns:
        model_words = row[model].split()
        h_ali, m_ali = align_pair(human_words, model_words)

        bin_idx = 0
        for h_w, m_w in zip(h_ali, m_ali):
            if h_w != "<eps>":
                if m_w != "<eps>" and m_w != h_w:
                    bins[bin_idx].append(m_w)
                bin_idx += 1

    trusted_bins = []
    for b in bins:
        counts = Counter(b)
        valid = [word for word, count in counts.items() if count >= threshold or word == b[0]]
        trusted_bins.append(list(set(valid)))

    return trusted_bins

## 5. WER Evaluation

We compare traditional WER (against a flat human reference) with Lattice WER (against sequential alternatives).

In [5]:
def calculate_total_wer(ref, hyp):
    ref_words = ref.split()
    hyp_words = hyp.split()
    dp = get_edit_matrix(ref_words, hyp_words)
    return int(dp[len(ref_words)][len(hyp_words)]), len(ref_words)

def calculate_lattice_wer(bins, hyp):
    hyp_words = hyp.split()
    n, m = len(bins), len(hyp_words)
    dp = np.zeros((n + 1, m + 1))
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if hyp_words[j-1] in bins[i-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return int(dp[n][m]), n

summary_data = []
for model in model_columns:
    total_std_err = 0
    total_lat_err = 0
    total_count = 0

    for _, row in df.iterrows():
        std_err, count = calculate_total_wer(row['Human'], row[model])
        lat_err, _ = calculate_lattice_wer(build_bins(row), row[model])

        total_std_err += std_err
        total_lat_err += lat_err
        total_count += count

    summary_data.append({
        'Model': model,
        'Standard WER': total_std_err / total_count if total_count > 0 else 0,
        'Lattice WER': total_lat_err / total_count if total_count > 0 else 0
    })

results_df = pd.DataFrame(summary_data)
results_df['Improvement (%)'] = ((results_df['Standard WER'] - results_df['Lattice WER']) / results_df['Standard WER'] * 100).round(2)
results_df

,Model,Standard WER,Lattice WER,Improvement (%)
0,Model H,0.028117,0.023227,17.39
1,Model i,0.003667,0.003667,0.00
2,Model k,0.085575,0.068460,20.00
3,Model l,0.086797,0.078240,9.86
4,Model m,0.165037,0.143032,13.33
5,Model n,0.106357,0.085575,19.54
